In [ ]:
!pip install -qU qdrant-client langchain-openai langchain-qdrant

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.8 MB/s eta 0:00:00


In [ ]:
# importing all libraries needed
import pandas as pd #for data processing
import getpass, os #for input password interface

from langchain_openai import OpenAIEmbeddings, ChatOpenAI #for embedding and LLM service
from uuid import uuid4 #for generate id with uuid format

from langchain_core.documents import Document #for document format that stored to vector database collection
from langchain_qdrant import QdrantVectorStore #for langchain - qdrant vector database connector
from qdrant_client import QdrantClient #for qdrant client set up
from qdrant_client.http.models import Distance, VectorParams #for distance method and configuration of vector parameters

In [ ]:
from google.colab import userdata

# input openai api key
if userdata.get('OPENAI_API_KEY'):
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
else:
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [ ]:
# input qdrant api key and url
if userdata.get('QDRANT_API_KEY'):
  api_key_qdrant = userdata.get('QDRANT_API_KEY')
else:
  api_key_qdrant = getpass.getpass("Enter API key for Qdrant: ")
print("QDRANT API KEY telah diinput")

if userdata.get('QDRANT_URL'):
  url_qdrant = userdata.get('QDRANT_URL')
else:
  url_qdrant = getpass.getpass("Enter URL for Qdrant: ")
print("QDRANT URL telah diinput")

QDRANT API KEY telah diinput
QDRANT URL telah diinput


In [ ]:
#define embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#define llm model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [ ]:
#define directory path of dataset
data_path = "/content/drive/MyDrive/purwadhika/Module 3/Capstone 3/imdb_top_1000.csv"

In [ ]:
#read dataset with pandas
df = pd.read_csv(data_path)
print(df.shape) #show total row and column with format (row, column)
df.head() #show first 5 record of dataset

(1000, 16)


,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [ ]:
print(df.isnull().sum())

Poster_Link        0
Series_Title       0
Released_Year      0
Certificate      101
Runtime            0
Genre              0
IMDB_Rating        0
Overview           0
Meta_score       157
Director           0
Star1              0
Star2              0
Star3              0
Star4              0
No_of_Votes        0
Gross            169
dtype: int64


In [ ]:
data = df.copy() #copy data table
data = data.iloc[:500,].reset_index(drop=True)
data = data.drop_duplicates(subset=['Series_Title', 'Released_Year', 'Overview', 'Genre', 'IMDB_Rating', 'Director', 'Star1', 'Star2']).reset_index(drop=True) #delete duplicate rows
data.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [ ]:
data = data.dropna(subset=['Certificate', 'Meta_score', 'Gross']).reset_index(drop=True)

In [ ]:
print(data.isnull().sum())

Poster_Link      0
Series_Title     0
Released_Year    0
Certificate      0
Runtime          0
Genre            0
IMDB_Rating      0
Overview         0
Meta_score       0
Director         0
Star1            0
Star2            0
Star3            0
Star4            0
No_of_Votes      0
Gross            0
dtype: int64


In [ ]:
from re import S
#looping and set the data into Document format
documents = []
for i in range(data.shape[0]):
  Series_Title = data["Series_Title"][i]
  Genre = data["Genre"][i]
  IMDB_Rating  = data["IMDB_Rating"][i]
  Overview = data["Overview"][i]
  Meta_score = data["Meta_score"][i]
  Released_Year = data["Released_Year"][i]
  Runtime = data["Runtime"][i]
  Director = data["Director"][i]
  Star1 = data["Star1"][i]
  Star2 = data["Star2"][i]
  Star3 = data["Star3"][i]
  Star4 = data["Star4"][i]

  doc = Document(
      page_content=f"{Series_Title}\n{Genre}\n{IMDB_Rating}\n{Overview}\n{Meta_score}\n{Released_Year}\n{Runtime}\n{Director}\n{Star1}\n{Star2}\n{Star3}\n{Star4}",
      metadata={"title": str(Series_Title),"Relesased Year" : str (Released_Year), "IMDB Rating": str (IMDB_Rating)}
  )
  documents.append(doc)

#setup unique id
uuids = [str(uuid4()) for _ in range(len(documents))]

In [ ]:
documents[0]

Document(metadata={'title': 'The Shawshank Redemption', 'Relesased Year': '1994', 'IMDB Rating': '9.3'}, page_content='The Shawshank Redemption\nDrama\n9.3\nTwo imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.\n80.0\n1994\n142 min\nFrank Darabont\nTim Robbins\nMorgan Freeman\nBob Gunton\nWilliam Sadler')

In [ ]:
#save document to qdrant
qdrant = QdrantVectorStore.from_documents(
    documents,
    embeddings,
    url=url_qdrant,
    prefer_grpc=True,
    api_key=api_key_qdrant,
    collection_name="Movie_documents",
)

In [ ]:
#setup qdrant client
client = QdrantClient(
  url= url_qdrant,
  api_key = api_key_qdrant
)
#get all collection in your qdrant vector database
collections_response = client.get_collections()
print("Collections:", collections_response.collections)

Collections: [CollectionDescription(name='Movie_documents')]


In [ ]:
#qdrant vectorstore with specific collection name
qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name="Movie_documents",
    url=url_qdrant,
    api_key=api_key_qdrant
)

In [ ]:
#retrieve data from qdrant
results = qdrant.similarity_search(
    "the movie highes rating",
    k=2
)
results

[Document(metadata={'IMDB Rating': '8.7', 'title': "One Flew Over the Cuckoo's Nest", 'Relesased Year': '1975', '_id': '62307c15-b73d-4356-96ff-e3cfeb60a0ea', '_collection_name': 'Movie_documents'}, page_content="One Flew Over the Cuckoo's Nest\nDrama\n8.7\nA criminal pleads insanity and is admitted to a mental institution, where he rebels against the oppressive nurse and rallies up the scared patients.\n83.0\n1975\n133 min\nMilos Forman\nJack Nicholson\nLouise Fletcher\nMichael Berryman\nPeter Brocco"),
 Document(metadata={'title': 'Heat', 'Relesased Year': '1995', 'IMDB Rating': '8.2', '_id': '3b37b44f-2a77-40ad-adfe-847b5ce0c8dd', '_collection_name': 'Movie_documents'}, page_content='Heat\nCrime, Drama, Thriller\n8.2\nA group of professional bank robbers start to feel the heat from police when they unknowingly leave a clue at their latest heist.\n76.0\n1995\n170 min\nMichael Mann\nAl Pacino\nRobert De Niro\nVal Kilmer\nJon Voight')]